# تقسيم الأسنان العلوية الستة — دفتر التدريب (النسخة 2)

نموذج **U-Net بمشفّر MobileNetV2** (مدرّب مسبقاً على ImageNet) يقسّم الأسنان العلوية الست في صور أمامية بمباعد الشفاه إلى **7 أصناف**:
0 خلفية، و1–6 = **13، 12، 11، 21، 22، 23** (FDI، من يسار الصورة إلى يمينها). هذا الدفتر أنتج النموذج المستخدم في التطبيق (`docs/teeth_unet_v2.onnx`).

| النموذج | Dice تحقق | Dice اختبار | IoU اختبار | الحجم fp32 |
| --- | --- | --- | --- | --- |
| **MobileNetV2 (المعتمد)** | 0.9602 | 0.9562 (0.960 مع TTA) | 0.9165 | 16.5 MB |
| EfficientNet-B0 | 0.9571 | 0.9604 | 0.9239 | 19.4 MB |

الاختبار 20 صورة من نفس مصادر التدريب، فالرقم متفائل؛ الدقة على صور العيادة لم تُقَس بعد مقابل قياسات الفرجار.

## طريقتا التشغيل

| `MODE` | ماذا يفعل | يحتاج GPU؟ | المدة |
| --- | --- | --- | --- |
| `'evaluate'` | يقيّم النموذج المنشور (`teeth_unet_v2.onnx`) على صور الاختبار، واختبار التحمّل والدوران والتدوير التلقائي. **بلا تدريب.** | لا | ~5 دقائق |
| `'train'` | يدرّب MobileNetV2 وEfficientNet-B0 من الصفر بالإعدادات نفسها، ويختار، ويقيّم، ويصدّر ONNX | نعم (T4) | ~40 دقيقة |

**التشغيل:** اختر `MODE` و`GITHUB_REPO` في خلية الإعدادات، ثم **Runtime ← Run all**. (للتدريب: Runtime ← Change runtime type ← T4 GPU.)
البيانات تُحمَّل من مجلد `data/` في المستودع نفسه.

> نتائج التدريب من الصفر قد تختلف قليلاً في الخانة الثالثة (عشوائية GPU)؛ وضع `evaluate` يعيد أرقام النموذج المنشور بدقة.

In [ ]:
!pip -q install "segmentation-models-pytorch==0.5.0" albumentations onnx onnxruntime onnxscript

In [ ]:
# ===== الإعدادات =====
MODE = 'evaluate'                 # 'evaluate' = تقييم النموذج المنشور بلا تدريب | 'train' = تدريب من الصفر
GITHUB_REPO = 'USER/REPO'         # مستودعك على GitHub (مثلاً 'fawzi/smile-design') لتحميل البيانات والنموذج منه
BRANCH = 'main'

import os, glob, random, time, zipfile, json, urllib.request, shutil
import numpy as np, cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import onnxruntime as ort

assert MODE in ('evaluate', 'train'), "MODE يجب أن يكون 'evaluate' أو 'train'"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('MODE:', MODE, '| device:', DEVICE, '| torch', torch.__version__, '| smp', smp.__version__, '| albumentations', A.__version__)
if MODE == 'train' and DEVICE == 'cpu':
    print('⚠ لا يوجد GPU: فعّل T4 من Runtime ← Change runtime type، وإلا سيكون التدريب بطيئاً جداً.')

IMG = 384                       # حجم المدخل (ضغط مباشر إلى مربع، كما في التطبيق)
TEETH = ['13', '12', '11', '21', '22', '23']
NUM_CLASSES = 7
BATCH = 16
EPOCHS = 60
FREEZE_EPOCHS = 3
LR_ENC, LR_DEC = 1e-4, 5e-4     # معدل أقل للمشفّر المدرَّب مسبقاً وأعلى للمفكّك
WEIGHT_DECAY = 1e-4
DEC = (128, 64, 32, 16, 16)
ENCODERS = ['mobilenet_v2', 'efficientnet-b0']
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
SWAP = [0, 6, 5, 4, 3, 2, 1]    # تبديل أرقام الأسنان عند القلب الأفقي: 1<->6, 2<->5, 3<->4
OUT = 'outputs'; os.makedirs(OUT, exist_ok=True)

## 1) البيانات

`data/smile_teeth_v2.zip` في المستودع: صور أمامية بمباعد من ثلاث مجموعات مفتوحة على Roboflow (CC BY 4.0)، مع قناع PNG لكل صورة بقيم 0–6.
التدريب 287 (62 FDI-System + 43 tooth_frontal + 182 منخفضة الدقة)، التحقق 23، الاختبار 20. التفاصيل والمصادر في `data/README.md`.

In [ ]:
def fetch(rel_path, local_name):
    # يبحث عن الملف محلياً (عند تشغيل الدفتر من نسخة المستودع)، ثم يحمّله من GitHub، ثم يطلب رفعه
    for p in (local_name, os.path.join('..', rel_path), rel_path, os.path.join('/content', os.path.basename(rel_path))):
        if os.path.exists(p): return p
    if GITHUB_REPO and GITHUB_REPO != 'USER/REPO':
        url = f'https://github.com/{GITHUB_REPO}/raw/{BRANCH}/{rel_path}'
        print('تحميل', url)
        try:
            urllib.request.urlretrieve(url, local_name); return local_name
        except Exception as e:
            print('  تعذّر التحميل:', e)
    try:
        from google.colab import files
        print(f'ارفع الملف {os.path.basename(rel_path)} يدوياً:')
        up = files.upload()
        name = next(iter(up)); os.replace(name, local_name); return local_name
    except ImportError:
        raise FileNotFoundError(f'لم يُعثر على {rel_path}. ضع GITHUB_REPO الصحيح أو شغّل الدفتر من داخل مجلد training في المستودع.')

zip_path = fetch('data/smile_teeth_v2.zip', 'smile_teeth_v2.zip')
if not os.path.isdir('smile_teeth_v2'):
    with zipfile.ZipFile(zip_path) as z: z.extractall('.')
ROOT = os.path.abspath('smile_teeth_v2')

def collect(split):
    out = []
    for p in sorted(glob.glob(f'{ROOT}/{split}/images/*.jpg')):
        fid = os.path.splitext(os.path.basename(p))[0]
        out.append((p, f'{ROOT}/{split}/masks/{fid}.png'))
    return out

train_items, val_items, test_items = collect('train'), collect('val'), collect('test')
print('تدريب / تحقق / اختبار:', len(train_items), len(val_items), len(test_items))
assert (len(train_items), len(val_items), len(test_items)) == (287, 23, 20), 'عدد الصور لا يطابق smile_teeth_v2'

## 2) القراءة والتعزيز

**القلب الأفقي يبدّل الأرقام** (13↔23، 12↔22، 11↔21)، وإلا تعلّم النموذج ترتيباً خاطئاً.

**تعزيز يشبه صور العيادة:** دوران ±20° وتكبير 0.85–1.15 وإزاحة 6%، سطوع وتباين ±0.3، غاما، تغيير ألوان نحو الأصفر والبني (أسنان مصبوغة)، ضبابية، CLAHE، ضغط JPEG 25–90 (صور واتساب)، ضوضاء، وإضاءة غير متساوية.

In [ ]:
geo_color = A.Compose([
    A.Affine(rotate=(-20, 20), scale=(0.85, 1.15), translate_percent=(-0.06, 0.06), p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.RandomGamma(gamma_limit=(70, 140), p=0.4),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.6),
    A.RGBShift(r_shift_limit=15, g_shift_limit=12, b_shift_limit=20, p=0.4),
    A.GaussianBlur(blur_limit=(3, 7), p=0.25),
    A.CLAHE(p=0.2),
])
to_tensor = A.Compose([A.Normalize(), ToTensorV2()])

def degrade(img):
    h, w = img.shape[:2]
    if random.random() < 0.5:                                   # ضغط JPEG
        q = random.randint(25, 90)
        _, enc = cv2.imencode('.jpg', img[:, :, ::-1], [cv2.IMWRITE_JPEG_QUALITY, q])
        img = cv2.imdecode(enc, 1)[:, :, ::-1]
    if random.random() < 0.3:                                   # ضوضاء
        img = np.clip(img + np.random.normal(0, random.uniform(2, 8), img.shape), 0, 255)
    if random.random() < 0.3:                                   # إضاءة غير متساوية
        g = np.linspace(random.uniform(0.6, 1.0), random.uniform(1.0, 1.3), w)
        if random.random() < 0.5: g = g[::-1]
        img = np.clip(img * g[None, :, None], 0, 255)
    return np.ascontiguousarray(img.astype(np.uint8))

def load_pair(ip, mp):
    img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
    msk = cv2.imread(mp, 0)
    img = cv2.resize(img, (IMG, IMG), interpolation=cv2.INTER_AREA)
    msk = cv2.resize(msk, (IMG, IMG), interpolation=cv2.INTER_NEAREST)
    return img, msk

def flip_swap(img, msk):
    img = img[:, ::-1].copy(); msk = msk[:, ::-1].copy()
    return img, np.where(msk > 0, NUM_CLASSES - msk, 0).astype(np.uint8)

class TeethDS(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        img, msk = load_pair(*self.items[i])
        if self.train:
            if random.random() < 0.5: img, msk = flip_swap(img, msk)
            o = geo_color(image=img, mask=msk); img, msk = o['image'], o['mask']
            img = degrade(img)
        o = to_tensor(image=img, mask=msk)
        return o['image'], o['mask'].long()

def mk_loader(items, train):
    return DataLoader(TeethDS(items, train), batch_size=BATCH, shuffle=train, num_workers=2,
                      pin_memory=(DEVICE == 'cuda'), drop_last=train)

PALETTE = np.array([[0,0,0],[230,25,75],[245,130,48],[255,225,25],[60,180,75],[0,130,200],[145,30,180]], np.uint8)
def overlay(img, m, a=0.5):
    o = img.astype(np.float32); k = m > 0
    o[k] = (1 - a) * o[k] + a * PALETTE[m][k]; return o.astype(np.uint8)
def denorm(x):
    return (np.clip(x.permute(1, 2, 0).numpy() * STD + MEAN, 0, 1) * 255).astype(np.uint8)

# فحص: القلب مع التبديل يحافظ على الترتيب (13 يسار و23 يمين)
_i, _m = load_pair(*val_items[0]); _fi, _fm = flip_swap(_i, _m)
assert np.nonzero(_fm == 1)[1].mean() < np.nonzero(_fm == 6)[1].mean(), 'تبديل الأرقام عند القلب لا يعمل'

if MODE == 'train':
    train_dl, val_dl, test_dl = mk_loader(train_items, True), mk_loader(val_items, False), mk_loader(test_items, False)
    ds = TeethDS(train_items[:8], True)
    fig, axs = plt.subplots(2, 4, figsize=(16, 8))
    for ax, i in zip(axs.ravel(), range(8)):
        x, y = ds[i]; ax.imshow(overlay(denorm(x), y.numpy())); ax.axis('off')
    plt.suptitle('augmented training samples'); plt.tight_layout(); plt.show()

## 3) النموذج والخسارة والتقييم

Dice وIoU **مجمّعان على كل صور المجموعة** لكل سن (مجموع التقاطعات ÷ مجموع المساحات)، ثم المتوسط على الأسنان الست.
`predict_np` تعمل مع نموذج PyTorch أو مع ملف ONNX بالمعالجة نفسها التي يستخدمها التطبيق.

In [ ]:
def build_model(enc, pretrained=True):
    names = {'mobilenet_v2': ['mobilenet_v2'],
             'efficientnet-b0': ['efficientnet-b0', 'timm-efficientnet-b0', 'tu-efficientnet_b0']}[enc]
    last = None
    for n in names:
        try:
            return smp.Unet(encoder_name=n, encoder_weights='imagenet' if pretrained else None,
                            in_channels=3, classes=NUM_CLASSES, decoder_channels=DEC).to(DEVICE)
        except Exception as e:
            last = e
    raise last

dice_loss = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
ce_loss = nn.CrossEntropyLoss()
def criterion(lg, y): return dice_loss(lg, y) + ce_loss(lg, y)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    inter = np.zeros(NUM_CLASSES); ps = np.zeros(NUM_CLASSES); gs = np.zeros(NUM_CLASSES)
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE); p = model(x).argmax(1)
        for c in range(1, NUM_CLASSES):
            pc, yc = (p == c), (y == c)
            inter[c] += (pc & yc).sum().item(); ps[c] += pc.sum().item(); gs[c] += yc.sum().item()
    dice = 2 * inter / np.maximum(ps + gs, 1); iou = inter / np.maximum(ps + gs - inter, 1)
    return dice[1:], iou[1:]

def normalize_np(img_u8):              # مطابقة لـ A.Normalize() وللتطبيق
    return np.ascontiguousarray(((img_u8.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None], dtype=np.float32)

def softmax_np(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

BACKEND = {}                           # {'torch': model} أو {'onnx': InferenceSession}
def logits_np(x):
    if 'onnx' in BACKEND:
        s = BACKEND['onnx']; return s.run(None, {s.get_inputs()[0].name: x})[0]
    m = BACKEND['torch']; m.eval()
    with torch.no_grad(): return m(torch.from_numpy(x).to(DEVICE)).float().cpu().numpy()

def predict_prob(img_u8, tta=False):
    x = normalize_np(img_u8)
    pr = softmax_np(logits_np(x))
    if tta:
        pf = softmax_np(logits_np(np.ascontiguousarray(x[..., ::-1])))[..., ::-1][:, SWAP]
        pr = (pr + pf) / 2
    return pr[0]

def predict_np(img_u8, tta=False): return predict_prob(img_u8, tta).argmax(0)

def scores_items(items, tf=None, tta=False):
    inter = np.zeros(NUM_CLASSES); ps = np.zeros(NUM_CLASSES); gs = np.zeros(NUM_CLASSES)
    for ip, mp in items:
        img, msk = load_pair(ip, mp)
        if tf: img, msk = tf(img, msk)
        p = predict_np(img, tta)
        for c in range(1, NUM_CLASSES):
            pc, yc = (p == c), (msk == c)
            inter[c] += (pc & yc).sum(); ps[c] += pc.sum(); gs[c] += yc.sum()
    dice = (2 * inter / np.maximum(ps + gs, 1))[1:]; iou = (inter / np.maximum(ps + gs - inter, 1))[1:]
    return dice, iou

def dice_items(items, tf=None, tta=False): return float(scores_items(items, tf, tta)[0].mean())

def print_per_tooth(dice, iou, title):
    print(title)
    print('السن   ' + '  '.join(f'{t:>6s}' for t in TEETH) + '   المتوسط')
    print('Dice   ' + '  '.join(f'{v:6.3f}' for v in dice) + f'   {dice.mean():.3f}')
    print('IoU    ' + '  '.join(f'{v:6.3f}' for v in iou) + f'   {iou.mean():.3f}')

## 4) التدريب (وضع `train` فقط)

مرحلتان: أول 3 إيبوك بمشفّر مجمّد (مع تثبيت BatchNorm فيه)، ثم fine-tuning. AdamW بمعدلين (المشفّر 1e-4 والمفكّك 5e-4) مع Cosine، ودقة مختلطة (AMP)، والخسارة Dice + Cross-Entropy. يُحفظ أفضل إيبوك حسب Dice التحقق.

In [ ]:
def train_model(model, tr_dl, va_dl, epochs, lr_enc, lr_dec, freeze_epochs, ckpt):
    enc_ids = {id(p) for p in model.encoder.parameters()}
    rest = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW([{'params': model.encoder.parameters(), 'lr': lr_enc},
                             {'params': rest, 'lr': lr_dec}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    use_amp = DEVICE == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    best, hist = -1.0, []
    for ep in range(epochs):
        freeze = ep < freeze_epochs
        for p in model.encoder.parameters(): p.requires_grad = not freeze
        model.train()
        if freeze: model.encoder.eval()
        t0, run, n = time.time(), 0.0, 0
        for x, y in tr_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                lg = model(x)
            loss = criterion(lg.float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * x.size(0); n += x.size(0)
        sched.step()
        d, _ = evaluate(model, va_dl); m = float(d.mean()); hist.append(m)
        tag = ''
        if m > best:
            best = m; torch.save(model.state_dict(), ckpt); tag = '  <- أفضل'
        print(f'epoch {ep+1:02d}/{epochs} {"[مجمّد]" if freeze else "       "} loss {run/n:.4f} | val dice {m:.4f} | {time.time()-t0:.0f}s{tag}')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return best, hist

## 5) تدريب المشفّرين ومقارنتهما — أو تحميل النموذج المنشور

الاختيار بحسب Dice **التحقق** (وليس الاختبار). صغر مجموعة التحقق (23 صورة) يعني أن فرقاً أقل من 0.01 لا يُعدّ حاسماً، وعندها يُفضَّل الأصغر حجماً (أسرع على جوال الطبيب).

In [ ]:
RESULTS = {'mode': MODE}
if MODE == 'train':
    results = {}
    for enc in ENCODERS:
        print(f'\n===== {enc} =====')
        torch.manual_seed(SEED)
        m = build_model(enc)
        n_params = sum(p.numel() for p in m.parameters())
        print(f'params {n_params/1e6:.2f}M (~{n_params*4/1e6:.1f} MB fp32)')
        best, hist = train_model(m, train_dl, val_dl, EPOCHS, LR_ENC, LR_DEC, FREEZE_EPOCHS, f'{OUT}/best_{enc.replace("-", "_")}.pt')
        d, i = evaluate(m, test_dl)
        results[enc] = dict(val=best, test_dice=float(d.mean()), test_iou=float(i.mean()),
                            params=n_params, size_mb=n_params * 4 / 1e6, hist=hist, per_tooth=d.tolist())
        del m
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    print('\nالنموذج          Dice تحقق  Dice اختبار  IoU اختبار  الحجم fp32')
    for k, r in results.items():
        print(f'{k:16s} {r["val"]:.4f}     {r["test_dice"]:.4f}      {r["test_iou"]:.4f}     {r["size_mb"]:.1f} MB')
    plt.figure(figsize=(8, 4))
    for k, r in results.items(): plt.plot(range(1, len(r['hist']) + 1), r['hist'], label=k)
    plt.xlabel('epoch'); plt.ylabel('validation Dice (6 teeth)'); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(f'{OUT}/training_curves.png', dpi=120); plt.show()

    WINNER = max(results, key=lambda k: results[k]['val'])
    close = [k for k in results if results[WINNER]['val'] - results[k]['val'] < 0.01]
    if len(close) > 1:
        WINNER = min(close, key=lambda k: results[k]['size_mb'])
    print('المختار:', WINNER)
    model = build_model(WINNER, pretrained=False)
    model.load_state_dict(torch.load(f'{OUT}/best_{WINNER.replace("-", "_")}.pt', map_location=DEVICE))
    BACKEND.clear(); BACKEND['torch'] = model
    RESULTS['encoders'] = {k: {kk: vv for kk, vv in r.items() if kk != 'hist'} for k, r in results.items()}
    RESULTS['chosen'] = WINNER
else:
    onnx_file = fetch('docs/teeth_unet_v2.onnx', 'teeth_unet_v2.onnx')
    sess = ort.InferenceSession(onnx_file, providers=['CPUExecutionProvider'])
    inp, out = sess.get_inputs()[0], sess.get_outputs()[0]
    print(f'النموذج المنشور: {os.path.getsize(onnx_file)/1e6:.1f} MB | مدخل {inp.name} {inp.shape} | مخرج {out.name} {out.shape}')
    assert list(inp.shape)[1:] == [3, IMG, IMG], 'حجم مدخل النموذج لا يطابق IMG'
    BACKEND.clear(); BACKEND['onnx'] = sess
    RESULTS['chosen'] = 'mobilenet_v2 (published ONNX)'

d, i = scores_items(test_items)
d_tta, _ = scores_items(test_items, tta=True)
print_per_tooth(d, i, f'\nالاختبار ({len(test_items)} صورة، بدون TTA):')
print(f'مع TTA: Dice {d_tta.mean():.4f}')
RESULTS['test'] = dict(dice=float(d.mean()), iou=float(i.mean()), dice_tta=float(d_tta.mean()),
                       per_tooth_dice=dict(zip(TEETH, d.round(4).tolist())), per_tooth_iou=dict(zip(TEETH, i.round(4).tolist())))

fig, axs = plt.subplots(2, 4, figsize=(16, 7))
for col, (ip, mp) in enumerate(test_items[:4]):
    img, msk = load_pair(ip, mp)
    axs[0, col].imshow(overlay(img, msk)); axs[0, col].set_title('ground truth'); axs[0, col].axis('off')
    axs[1, col].imshow(overlay(img, predict_np(img, tta=True).astype(np.uint8))); axs[1, col].set_title('prediction (TTA)'); axs[1, col].axis('off')
plt.tight_layout(); plt.show()

## 6) اختبار التحمّل (صور الاختبار) مع TTA وبدونه

كيف يتصرف النموذج عندما تسوء ظروف التصوير؟ هذه الأرقام أصدق للتقرير من رقم واحد على صور نظيفة.
عند الدوران يُملأ ما يخرج عن الصورة بالأسود في الصورة والقناع معاً.

In [ ]:
def tf_rot(a):
    def f(img, m):
        h, w = img.shape[:2]; M = cv2.getRotationMatrix2D((w/2, h/2), a, 1)
        return (cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0),
                cv2.warpAffine(m, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0))
    return f
def tf_dark(img, m): return (img * 0.5).astype(np.uint8), m
def tf_blur(img, m): return cv2.GaussianBlur(img, (9, 9), 0), m
def tf_warm(img, m):
    o = img.astype(np.int16); o[..., 0] += 25; o[..., 2] -= 25; return np.clip(o, 0, 255).astype(np.uint8), m
def tf_jpeg(img, m):
    _, e = cv2.imencode('.jpg', img[:, :, ::-1], [cv2.IMWRITE_JPEG_QUALITY, 30]); return cv2.imdecode(e, 1)[:, :, ::-1].copy(), m

tests = {'نظيفة': None, 'دوران +15': tf_rot(15), 'دوران -15': tf_rot(-15), 'إضاءة منخفضة': tf_dark,
         'ضبابية': tf_blur, 'دفء لوني': tf_warm, 'JPEG جودة 30': tf_jpeg}
RESULTS['robustness'] = {}
print('الحالة            بدون TTA   مع TTA')
for k, tf in tests.items():
    a, b = dice_items(test_items, tf, False), dice_items(test_items, tf, True)
    RESULTS['robustness'][k] = dict(plain=a, tta=b)
    print(f'{k:16s} {a:.3f}      {b:.3f}')

## 7) منحنى الدقة مقابل ميل الصورة

الزاوية التي ينزل عندها Dice تحت نحو 0.90 هي حدّ التحمّل العملي، وتُكتب في دليل التصوير للطبيب («لا تُمِل الجوال أكثر من ... درجة»).

In [ ]:
ANGLES = [-45, -30, -20, -10, -5, 0, 5, 10, 20, 30, 45]
curve = {}
for a in ANGLES:
    d, _ = scores_items(test_items, tf_rot(a))
    curve[a] = float(d.mean())
    print(f'{a:+4d}°   متوسط Dice {d.mean():.3f}   أضعف سن {TEETH[int(d.argmin())]} ({d.min():.3f})')
RESULTS['rotation_curve'] = {str(a): v for a, v in curve.items()}
plt.figure(figsize=(7, 4))
plt.plot(ANGLES, [curve[a] for a in ANGLES], marker='o'); plt.axhline(0.90, color='gray', linestyle='--', linewidth=1)
plt.xlabel('rotation angle (degrees)'); plt.ylabel('mean Dice (6 teeth)'); plt.title('Robustness to in-plane rotation')
plt.ylim(0, 1); plt.grid(alpha=0.3); plt.savefig(f'{OUT}/rotation_curve.png', dpi=120); plt.show()

## 8) التدوير التلقائي (الاتجاهات الأربعة)

لكل صورة تُجرَّب 4 اتجاهات، ودرجة كل اتجاه = **عدد الأسنان الموجودة + الثقة + نقطتان إن كان ترتيب الأسنان سليماً** (13 ثم 12 … ثم 23 من اليسار لليمين)، ويُختار الأعلى. التطبيق يستخدم المنطق نفسه.
الاختبار: نقلب صور الاختبار عمداً بكل اتجاه ونرى هل يستعيد النظام الاتجاه الصحيح.

In [ ]:
MIN_FRAC = 300 / (320 * 320)          # أصغر مساحة لسن حتى يُعدّ موجوداً (نفس عتبة التطبيق)

def analyze(img_u8):
    prob = predict_prob(cv2.resize(img_u8, (IMG, IMG), interpolation=cv2.INTER_AREA))
    mask, conf_map = prob.argmax(0), prob.max(0)
    found = sum(int((mask == c).sum() >= MIN_FRAC * mask.size) for c in range(1, NUM_CLASSES))
    tooth = mask > 0
    conf = float(conf_map[tooth].mean()) if tooth.any() else 0.0
    xs = [np.nonzero(mask == c)[1].mean() if (mask == c).sum() >= MIN_FRAC * mask.size else None for c in range(1, NUM_CLASSES)]
    ordered = None not in xs and all(xs[i] < xs[i + 1] for i in range(5))
    return found, conf, ordered

def auto_orient(img_u8):
    best = None
    for k in range(4):
        found, conf, ordered = analyze(np.ascontiguousarray(np.rot90(img_u8, k)))
        score = found + conf + (2 if ordered else 0)
        if best is None or score > best[0]: best = (score, k)
    return best[1]

test_imgs = [cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB) for ip, _ in test_items]
RESULTS['auto_orient'] = {}
for k_true in range(4):
    hits = [auto_orient(np.ascontiguousarray(np.rot90(img, k_true))) == (4 - k_true) % 4 for img in test_imgs]
    RESULTS['auto_orient'][f'{k_true*90}deg'] = f'{sum(hits)}/{len(hits)}'
    print(f'صورة مدوّرة {k_true * 90:3d}°: استُعيد اتجاهها في {sum(hits)}/{len(hits)}')

## 9) التصدير إلى ONNX للتطبيق (وضع `train` فقط)

مدخل ثابت `1×3×384×384` باسم `image`، ومخرج `1×7×384×384` باسم `logits`، في ملف واحد. التطبيق يطبّق softmax وTTA بنفسه.
يُتحقق من مطابقة ONNX لـ PyTorch على صور الاختبار قبل اعتماده.

In [ ]:
if MODE == 'train':
    import onnx
    onnx_path = f'{OUT}/teeth_unet_v2.onnx'
    m_cpu = build_model(WINNER, pretrained=False).to('cpu')
    m_cpu.load_state_dict(torch.load(f'{OUT}/best_{WINNER.replace("-", "_")}.pt', map_location='cpu')); m_cpu.eval()
    dummy = torch.randn(1, 3, IMG, IMG)
    errors = []
    for extra in (dict(dynamo=False, opset_version=17), dict(opset_version=17), dict(dynamo=True, opset_version=18, external_data=False)):
        try:
            torch.onnx.export(m_cpu, dummy, onnx_path, input_names=['image'], output_names=['logits'], **extra)
            print('طريقة التصدير:', extra); break
        except Exception as e:
            errors.append(f'{extra}: {type(e).__name__}: {str(e)[:200]}')
    else:
        raise RuntimeError('فشل التصدير:\n' + '\n'.join(errors))
    onnx.save_model(onnx.load(onnx_path), onnx_path, save_as_external_data=False)   # ملف واحد
    for f in glob.glob(onnx_path + '.*'): os.remove(f)
    onnx.checker.check_model(onnx.load(onnx_path))

    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    max_diff, agree = 0.0, []
    for ip, mp in test_items[:5]:
        x = normalize_np(load_pair(ip, mp)[0])
        with torch.no_grad(): ref = m_cpu(torch.from_numpy(x)).numpy()
        o = sess.run(None, {'image': x})[0]
        assert o.shape == (1, NUM_CLASSES, IMG, IMG), o.shape
        max_diff = max(max_diff, float(np.abs(o - ref).max())); agree.append(float((o.argmax(1) == ref.argmax(1)).mean()))
    size_mb = os.path.getsize(onnx_path) / 1e6
    print(f'ONNX {size_mb:.1f} MB | أكبر فرق عن PyTorch {max_diff:.1e} | اتفاق التنبؤ {np.mean(agree):.2%}')
    assert max_diff < 1e-2 and np.mean(agree) > 0.999, 'ملف ONNX لا يطابق PyTorch'
    RESULTS['onnx'] = dict(size_mb=size_mb, max_abs_diff=max_diff, argmax_agreement=float(np.mean(agree)))
else:
    print('وضع evaluate: لا تصدير (النموذج المنشور مستخدم كما هو).')

In [ ]:
with open(f'{OUT}/results.json', 'w', encoding='utf-8') as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=1, default=float)
print('النتائج محفوظة في', f'{OUT}/results.json')
for f in sorted(os.listdir(OUT)): print(f'  {f:28s} {os.path.getsize(os.path.join(OUT, f))/1e6:7.2f} MB')
if MODE == 'train':
    try:
        from google.colab import files
        files.download(f'{OUT}/teeth_unet_v2.onnx')
    except ImportError:
        pass

## بعد التشغيل

- **وضع `evaluate`:** يجب أن يعطي Dice الاختبار ≈ 0.956 (0.960 مع TTA). هذا يؤكد أن ملف التطبيق هو النموذج الموثّق.
- **وضع `train`:** `outputs/teeth_unet_v2.onnx` يحل محل `docs/teeth_unet_v2.onnx` فقط إن كانت نتائجه مثل المنشور أو أفضل. و`best_mobilenet_v2.pt` أوزان PyTorch.
- `results.json` و`training_curves.png` و`rotation_curve.png` للتقرير.

> صور العيادة ليست في هذا الدفتر. تدريب النموذج عليها خطوة لاحقة تحتاج موافقة المرضى وأقنعة يراجعها إنسان، ولا تُرفع إلى المستودع.